# MediQuest Triage Desk — live run
An agent proposes a **South African Triage Scale** priority from vitals + the presenting complaint, cites every row it used, and **stops**. A named nurse confirms, overrides or rejects. Open-weights model (Qwen 2.5 via Ollama, on this GPU), LangGraph, our `triage-desk` MCP server + the borrowed `mcp-server-time`. Synthetic patients only — not for clinical use.

Repo: https://github.com/aladros/mediquest-triage-desk

In [ ]:
%%bash
# setup: Ollama + open-weights model + the app (~2 min on a T4)
set -e
(apt-get -qq update && apt-get -qq install -y pciutils lshw zstd) >/dev/null 2>&1 || true
curl -fsSL https://ollama.com/install.sh | sh >/dev/null 2>&1
nohup ollama serve > /tmp/ollama.log 2>&1 &
sleep 5
ollama pull qwen2.5:7b >/dev/null 2>&1
rm -rf /kaggle/working/app && git clone -q https://github.com/aladros/mediquest-triage-desk /kaggle/working/app
pip install -q uv -e /kaggle/working/app
ollama list

In [ ]:
import os, json
os.chdir('/kaggle/working/app'); os.environ['TRIAGE_MODEL'] = 'qwen2.5:7b'
from mediquest_triage_desk.notebook import TriageDesk
patients = {p['patient_id']: p for p in json.load(open('data/demo_patients.json'))}
desk = await TriageDesk.open()

### 1 · Chest pain, mid-range vitals

In [ ]:
await desk.triage(patients['OPD-0412'])

In [ ]:
await desk.decide('Nurse Achieng', 'confirm')

### 2 · Drowsy, febrile, on a stretcher

In [ ]:
await desk.triage(patients['OPD-0414'])

In [ ]:
await desk.decide('Nurse Achieng', 'confirm')

### 3 · Routine refill

In [ ]:
await desk.triage(patients['OPD-0413'])

In [ ]:
await desk.decide('Nurse Achieng', 'confirm')

### 4 · A child — adult chart does not apply

In [ ]:
await desk.triage(patients['OPD-0416'])

### 5 · Where it fails: a denied symptom

In [ ]:
await desk.triage({'patient_id': 'OPD-0418', 'age_years': 33,
    'complaint': 'No chest pain and no shortness of breath. Itchy rash on both arms for a week.',
    'vitals': {'respiratory_rate': 14, 'heart_rate': 76, 'systolic_bp': 122, 'temperature_c': 36.6,
               'avpu': 'alert', 'mobility': 'walking', 'trauma': False}})

In [ ]:
await desk.decide('Nurse Achieng', 'override')   # the gate refuses an override with no colour/reason

In [ ]:
await desk.decide('Nurse Achieng', 'override', colour='green', reason='patient denies chest pain; rash only')

### 6 · 45 minutes later: the refill patient feels faint — re-check

In [ ]:
recheck = json.load(open('data/demo_recheck.json'))
await desk.recheck(recheck['patient_id'], recheck['vitals'])

In [ ]:
await desk.decide('Nurse Achieng', 'confirm')

### Queue (ordered only by nurse-confirmed priorities) and the audit trail

In [ ]:
await desk.queue()

In [ ]:
!tail -n 3 var/audit.jsonl | cut -c1-300